# Transfermarkt World Cup Lineup Scraper

This notebook scrapes World Cup match lineups from Transfermarkt to generate quiz data for Starting11.

**Approach:**
1. Scrape the lineup page to get 11 players + positions
2. For each player, scrape their profile to get career history
3. Match the match date to find which club they were at
4. Output quiz JSON + mock visualization

**Important:** Be cautious with rate limits! Add delays between requests.

In [1]:
print("hello")

hello


In [ ]:
# Install required packages if needed
!pip install requests beautifulsoup4 pandas lxml

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json
import time
import re
from datetime import datetime
from IPython.display import HTML, display

# Rate limiting configuration
REQUEST_DELAY = 3  # seconds between requests - be respectful!

# Headers to mimic a browser (required by Transfermarkt)
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5',
    'Accept-Encoding': 'gzip, deflate, br',
    'Connection': 'keep-alive',
}

BASE_URL = 'https://www.transfermarkt.com'

def fetch_page(url, delay=True):
    """Fetch a page with rate limiting and proper headers."""
    if delay:
        print(f"  Waiting {REQUEST_DELAY}s before request...")
        time.sleep(REQUEST_DELAY)
    
    print(f"  Fetching: {url}")
    response = requests.get(url, headers=HEADERS)
    
    if response.status_code != 200:
        print(f"  ERROR: Status code {response.status_code}")
        return None
    
    return BeautifulSoup(response.content, 'lxml')

print("Libraries loaded. Ready to scrape!")

Libraries loaded. Ready to scrape!


## Step 1: Scrape Match Lineup Page

We'll scrape the lineup/formation page to get the starting 11 players.

In [3]:
# Test match: 2022 World Cup Final - Argentina vs France
# Match ID from Transfermarkt: 3975879
MATCH_ID = '3975879'
MATCH_DATE = '2022-12-18'

# Lineup page URL
lineup_url = f"{BASE_URL}/spielbericht/aufstellung/spielbericht/{MATCH_ID}"
print(f"Target URL: {lineup_url}")

Target URL: https://www.transfermarkt.com/spielbericht/aufstellung/spielbericht/3975879


In [4]:
# Fetch the lineup page
soup = fetch_page(lineup_url, delay=False)  # First request, no delay needed

if soup:
    # Get page title to verify we got the right page
    title = soup.find('title')
    print(f"Page title: {title.text if title else 'No title found'}")
else:
    print("Failed to fetch page")

  Fetching: https://www.transfermarkt.com/spielbericht/aufstellung/spielbericht/3975879
Page title: Argentina - France, 18/12/2022 - World Cup 2022 - Statistics | Transfermarkt


In [5]:
def extract_lineup_from_page(soup, team_index=0):
    """
    Extract starting lineup from the page.
    team_index: 0 for home team, 1 for away team
    
    Returns list of dicts with player info.
    """
    players = []
    
    # Find all lineup containers (there should be 2 - one per team)
    # The formation graphic uses div elements with player info
    lineup_boxes = soup.find_all('div', class_='aufstellung-spieler-container')
    
    if not lineup_boxes:
        # Try alternative selector - the graphic might use different classes
        lineup_boxes = soup.find_all('div', {'class': re.compile(r'aufstellung')})
        print(f"Found {len(lineup_boxes)} elements with 'aufstellung' class")
    
    # Let's inspect the page structure
    # Find all links to player profiles in lineup sections
    lineup_sections = soup.find_all('div', class_='large-6')
    print(f"Found {len(lineup_sections)} large-6 divs")
    
    # Look for the formation tables
    tables = soup.find_all('table')
    print(f"Found {len(tables)} tables")
    
    # Try to find player links in the Starting Line-up section
    all_links = soup.find_all('a', href=re.compile(r'/[^/]+/profil/spieler/\d+'))
    print(f"Found {len(all_links)} player profile links")
    
    return all_links

# Test the extraction
if soup:
    links = extract_lineup_from_page(soup)
    for i, link in enumerate(links[:15]):  # Show first 15
        print(f"{i+1}. {link.text.strip()} -> {link.get('href', 'no href')}")

Found 0 elements with 'aufstellung' class
Found 6 large-6 divs
Found 63 tables
Found 50 player profile links
1.  -> /emiliano-martinez/profil/spieler/111873
2.  -> /cristian-romero/profil/spieler/355915
3.  -> /nicolas-otamendi/profil/spieler/54781
4.  -> /nicolas-tagliafico/profil/spieler/131225
5.  -> /nahuel-molina/profil/spieler/424042
6.  -> /enzo-fernandez/profil/spieler/648195
7.  -> /rodrigo-de-paul/profil/spieler/255901
8.  -> /alexis-mac-allister/profil/spieler/534033
9.  -> /angel-di-maria/profil/spieler/45320
10.  -> /lionel-messi/profil/spieler/28003
11.  -> /julian-alvarez/profil/spieler/576024
12.  -> /hugo-lloris/profil/spieler/17965
13.  -> /raphael-varane/profil/spieler/164770
14.  -> /dayot-upamecano/profil/spieler/344695
15.  -> /theo-hernandez/profil/spieler/339808


In [6]:
def parse_lineup_page(soup):
    """
    Parse the lineup page to extract both teams' starting 11.
    Returns dict with home and away team lineups.
    """
    result = {
        'home': {'team': '', 'formation': '', 'players': []},
        'away': {'team': '', 'formation': '', 'players': []}
    }
    
    # Find team names from the header
    team_links = soup.find_all('a', class_='sb-vereinslink')
    if len(team_links) >= 2:
        result['home']['team'] = team_links[0].text.strip()
        result['away']['team'] = team_links[1].text.strip()
    
    # Find formation info - usually in headers like "Starting Line-up: 4-3-3"
    formation_headers = soup.find_all(text=re.compile(r'Starting Line-up:.*\d-\d-\d'))
    for i, header in enumerate(formation_headers[:2]):
        match = re.search(r'(\d-\d-\d(?:-\d)?)', str(header))
        if match:
            if i == 0:
                result['home']['formation'] = match.group(1)
            else:
                result['away']['formation'] = match.group(1)
    
    # Find all starting lineup sections
    # Looking for the visual formation graphic divs
    # Each player in the graphic has their number and name
    
    # The page structure has two main lineup sections
    # Let's look for all player entries with position info
    
    # Find all player entries in tables
    player_rows = soup.find_all('td', class_='hauptlink')
    print(f"Found {len(player_rows)} hauptlink cells")
    
    return result

if soup:
    result = parse_lineup_page(soup)
    print(f"Home team: {result['home']['team']} ({result['home']['formation']})")
    print(f"Away team: {result['away']['team']} ({result['away']['formation']})")

Found 0 hauptlink cells
Home team: Argentina ()
Away team: France ()


/var/folders/yg/2p4fh2yx63z296p0tjwgw1hc0000gn/T/ipykernel_37025/1390671896.py:18: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  formation_headers = soup.find_all(text=re.compile(r'Starting Line-up:.*\d-\d-\d'))


## Step 2: Debug - Let's look at the raw HTML structure

Since the page structure might be different than expected, let's inspect it more carefully.

In [ ]:
# Let's save the HTML to inspect it manually
if soup:
    # Save raw HTML for inspection
    with open('debug_lineup_page.html', 'w', encoding='utf-8') as f:
        f.write(soup.prettify())
    print("Saved HTML to debug_lineup_page.html")
    
    # Print some key sections
    print("\n--- Looking for lineup-related classes ---")
    for div in soup.find_all('div')[:100]:
        classes = div.get('class', [])
        if any('aufstellung' in c.lower() or 'lineup' in c.lower() or 'formation' in c.lower() 
               for c in classes):
            print(f"Found div with classes: {classes}")

In [ ]:
# Let's look at the structure more systematically
if soup:
    # Find all unique CSS classes that might be relevant
    all_classes = set()
    for tag in soup.find_all(True):
        classes = tag.get('class', [])
        for c in classes:
            if any(keyword in c.lower() for keyword in ['spiel', 'lineup', 'aufstellung', 'player', 'formation']):
                all_classes.add(c)
    
    print("Relevant CSS classes found:")
    for c in sorted(all_classes):
        print(f"  - {c}")

In [14]:
# IMPROVED VERSION - handles club name in images/titles/URLs
def get_player_club_at_date_v2(player_id, match_date, player_name=""):
    """
    Scrape a player's profile to find which club they were at on match_date.
    Improved version that checks multiple sources for club name.
    """
    profile_url = f"{BASE_URL}/anyplayer/profil/spieler/{player_id}"
    
    print(f"  Getting club for {player_name} (ID: {player_id})...")
    
    soup = fetch_page(profile_url)
    if not soup:
        return {'club': 'Unknown', 'club_id': None}
    
    # Find club link
    club_link = soup.find('a', href=re.compile(r'/[\w-]+/startseite/verein/\d+'))
    
    if club_link:
        club_href = club_link.get('href', '')
        club_match = re.search(r'/verein/(\d+)', club_href)
        club_id = club_match.group(1) if club_match else None
        
        club_name = None
        
        # Method 1: Direct text
        if club_link.text.strip():
            club_name = club_link.text.strip()
        
        # Method 2: Title attribute on link
        if not club_name and club_link.get('title'):
            club_name = club_link.get('title').strip()
        
        # Method 3: Image alt/title inside link
        if not club_name:
            img = club_link.find('img')
            if img:
                club_name = img.get('alt', '').strip() or img.get('title', '').strip()
        
        # Method 4: Extract from URL slug (e.g., "inter-miami-cf" -> "Inter Miami Cf")
        if not club_name:
            url_match = re.search(r'/([a-zA-Z0-9-]+)/startseite/verein/', club_href)
            if url_match:
                club_name = url_match.group(1).replace('-', ' ').title()
        
        if club_name:
            return {'club': club_name, 'club_id': club_id}
    
    return {'club': 'Unknown', 'club_id': None}

# Test with Messi
test_result = get_player_club_at_date_v2('28003', '2022-12-18', 'Lionel Messi')
print(f"\nResult: {test_result}")

  Getting club for Lionel Messi (ID: 28003)...
  Waiting 3s before request...
  Fetching: https://www.transfermarkt.com/anyplayer/profil/spieler/28003

Result: {'club': 'Inter Miami CF', 'club_id': '69261'}


In [15]:
# V3 - Get club at SPECIFIC DATE using transfer history
from datetime import datetime

def get_club_at_date(player_id, match_date_str, player_name=""):
    """
    Get the club a player was at on a specific date by parsing transfer history.
    
    Args:
        player_id: Transfermarkt player ID
        match_date_str: Date in 'YYYY-MM-DD' format (e.g., '2022-12-18')
        player_name: For logging
    
    Returns:
        dict with 'club' and 'club_id'
    """
    # Transfers page URL
    transfers_url = f"{BASE_URL}/a/transfers/spieler/{player_id}"
    
    print(f"  Getting transfer history for {player_name} (ID: {player_id})...")
    
    soup = fetch_page(transfers_url)
    if not soup:
        return {'club': 'Unknown', 'club_id': None}
    
    # Parse the target date
    match_date = datetime.strptime(match_date_str, '%Y-%m-%d')
    
    # Find all transfer rows - look for links with transfer dates
    # Transfer history has rows with date, club left, club joined
    transfers = []
    
    # Try to find the transfer history table/grid
    # Look for sections with transfer data - dates are in DD/MM/YYYY format
    
    # Method 1: Find all date-like text and associated club info
    # The transfer rows typically have: Season | Date | Left | Joined | MV | Fee
    
    # Find all elements that look like dates (DD/MM/YYYY)
    date_pattern = re.compile(r'(\d{2}/\d{2}/\d{4})')
    
    # Look for table rows or grid sections
    rows = soup.find_all(['tr', 'section'])
    
    for row in rows:
        text = row.get_text()
        date_match = date_pattern.search(text)
        
        if date_match:
            date_str = date_match.group(1)
            try:
                transfer_date = datetime.strptime(date_str, '%d/%m/%Y')
            except:
                continue
            
            # Find club links in this row - the "Joined" club
            club_links = row.find_all('a', href=re.compile(r'/transfers/verein/\d+'))
            
            # Usually there are 2 club links: "Left" and "Joined"
            # We want the "Joined" club (second one)
            if len(club_links) >= 2:
                joined_link = club_links[1]  # Second link is "Joined"
                club_name = joined_link.get('title', '') or joined_link.text.strip()
                
                # Extract club ID from href
                href = joined_link.get('href', '')
                club_id_match = re.search(r'/verein/(\d+)', href)
                club_id = club_id_match.group(1) if club_id_match else None
                
                transfers.append({
                    'date': transfer_date,
                    'club': club_name,
                    'club_id': club_id
                })
    
    # Sort transfers by date (oldest first)
    transfers.sort(key=lambda x: x['date'])
    
    print(f"    Found {len(transfers)} transfers")
    for t in transfers:
        print(f"      {t['date'].strftime('%Y-%m-%d')}: Joined {t['club']}")
    
    # Find the club at match_date
    # The player was at the club they "Joined" in the most recent transfer BEFORE match_date
    club_at_date = None
    for t in transfers:
        if t['date'] <= match_date:
            club_at_date = t
        else:
            break  # Past the match date, stop
    
    if club_at_date:
        print(f"    → On {match_date_str}: {club_at_date['club']}")
        return {'club': club_at_date['club'], 'club_id': club_at_date['club_id']}
    
    # Fallback: if no transfers found before date, they might still be at first club
    # or we couldn't parse the data
    return {'club': 'Unknown', 'club_id': None}

# Test with Messi for WC 2022 Final date
result = get_club_at_date('28003', '2022-12-18', 'Lionel Messi')
print(f"\nFinal result: {result}")

  Getting transfer history for Lionel Messi (ID: 28003)...
  Waiting 3s before request...
  Fetching: https://www.transfermarkt.com/a/transfers/spieler/28003
    Found 0 transfers

Final result: {'club': 'Unknown', 'club_id': None}


In [16]:
# DEBUG - Let's look at the actual HTML structure of the transfers page
transfers_url = f"{BASE_URL}/lionel-messi/transfers/spieler/28003"
print(f"Fetching: {transfers_url}")

debug_soup = fetch_page(transfers_url, delay=False)

if debug_soup:
    # Save full HTML for manual inspection
    with open('debug_messi_transfers.html', 'w', encoding='utf-8') as f:
        f.write(debug_soup.prettify())
    print("Saved to debug_messi_transfers.html")
    
    # Look for common transfer table patterns
    print("\n--- Looking for tables ---")
    tables = debug_soup.find_all('table')
    print(f"Found {len(tables)} tables")
    
    print("\n--- Looking for grid/responsive-table classes ---")
    grids = debug_soup.find_all(class_=re.compile(r'grid|responsive|transfer'))
    print(f"Found {len(grids)} grid-like elements")
    for g in grids[:5]:
        print(f"  Class: {g.get('class')}")
    
    print("\n--- Looking for date patterns (DD/MM/YYYY) ---")
    date_pattern = re.compile(r'\d{2}/\d{2}/\d{4}')
    all_text = debug_soup.get_text()
    dates_found = date_pattern.findall(all_text)
    print(f"Found {len(dates_found)} dates: {dates_found[:10]}")
    
    print("\n--- Looking for 'PSG' text ---")
    psg_elements = debug_soup.find_all(string=re.compile(r'PSG'))
    print(f"Found {len(psg_elements)} elements mentioning PSG")
    
    print("\n--- Page title ---")
    title = debug_soup.find('title')
    print(f"Title: {title.text if title else 'No title'}")
else:
    print("Failed to fetch page")

Fetching: https://www.transfermarkt.com/lionel-messi/transfers/spieler/28003
  Fetching: https://www.transfermarkt.com/lionel-messi/transfers/spieler/28003
Saved to debug_messi_transfers.html

--- Looking for tables ---
Found 0 tables

--- Looking for grid/responsive-table classes ---
Found 2 grid-like elements
  Class: ['tm-discover__hero-container', 'tm-discover__hero-container-grid', 'tm-discover-container-grid']
  Class: ['tm-footer__mobile', 'tm-footer__transfermarkt-info']

--- Looking for date patterns (DD/MM/YYYY) ---
Found 4 dates: ['15/07/2023', '31/12/2028', '24/06/1987', '11/12/2025']

--- Looking for 'PSG' text ---
Found 0 elements mentioning PSG

--- Page title ---
Title: Lionel Messi - Transfer history | Transfermarkt


In [17]:
# Check for embedded JSON or web component data
if debug_soup:
    print("--- Looking for script tags with JSON data ---")
    scripts = debug_soup.find_all('script')
    print(f"Found {len(scripts)} script tags")
    
    for i, script in enumerate(scripts):
        text = script.string or ''
        if 'transfer' in text.lower() or 'PSG' in text or 'Barcelona' in text:
            print(f"\nScript {i} contains relevant data:")
            print(text[:500] if len(text) > 500 else text)
    
    print("\n--- Looking for tm-player-transfer-history component ---")
    tm_component = debug_soup.find('tm-player-transfer-history')
    if tm_component:
        print("Found component!")
        print(f"Attributes: {dict(tm_component.attrs)}")
        # Check if translations attribute has useful data
        translations = tm_component.get('translations', '')
        if translations:
            print(f"\nTranslations attr: {translations[:200]}...")
    else:
        print("Component not found")
    
    print("\n--- Looking for any element with 'Barcelona' ---")
    barca = debug_soup.find_all(string=re.compile(r'Barcelona', re.I))
    print(f"Found {len(barca)} elements with 'Barcelona'")
    
    print("\n--- Check for data attributes ---")
    data_elements = debug_soup.find_all(attrs={'data-transfer': True})
    print(f"Found {len(data_elements)} elements with data-transfer attr")
    
    # Also check for any JSON in the page
    print("\n--- Looking for JSON-like structures ---")
    import json
    for script in scripts:
        text = script.string or ''
        if text.strip().startswith('{') or '__NEXT_DATA__' in text or 'window.' in text:
            print(f"Potential JSON script found (first 300 chars):")
            print(text[:300])
            print("...")

--- Looking for script tags with JSON data ---
Found 41 script tags

Script 3 contains relevant data:

    (function() {
        var cpBaseUrl = 'https://cp.transfermarkt.com';
        var cpController = cpBaseUrl + '/now.js';
        var cpPropertyId = '7a84b340';

        !function(C,o,n,t,P,a,s){C['CPObject']=n;C[n]||(C[n]=function(){
        (C[n].q=C[n].q||[]).push(arguments)});C[n].l=+new Date;a=o.createElement(t);
        s=o.getElementsByTagName(t)[0];a.src=P;s.parentNode.insertBefore(a,s)}
        (window,document,'cp','script',cpController);

        !function(C,o,n,t,P){if(!C[n].patch

Script 12 contains relevant data:

/*<![CDATA[*/
console.info("%c [TM-ADs] Initialize Ads on domain .com (spieler/transfers)", "background: #282828; color: #bada55")
console.info("%c [TM-ADs] Tisoomi is active -> add Tisoomi script", "background: #282828; color: #bada55")
console.info("%c [TM-ADs] Render slot rectangle1 (/58778164/d_side_1) for google", "background: #282828; color: #bada55")
c

In [18]:
# Try the career stats page - this often has club history in a regular table
career_url = f"{BASE_URL}/lionel-messi/leistungsdatendetails/spieler/28003"
print(f"Fetching career stats: {career_url}")

career_soup = fetch_page(career_url, delay=True)

if career_soup:
    # Save for inspection
    with open('debug_messi_career.html', 'w', encoding='utf-8') as f:
        f.write(career_soup.prettify())
    print("Saved to debug_messi_career.html")
    
    print("\n--- Looking for tables ---")
    tables = career_soup.find_all('table')
    print(f"Found {len(tables)} tables")
    
    print("\n--- Looking for club names ---")
    clubs = ['PSG', 'Barcelona', 'Inter Miami', 'Miami']
    for club in clubs:
        found = career_soup.find_all(string=re.compile(club, re.I))
        print(f"  '{club}': {len(found)} matches")
    
    print("\n--- Looking for season patterns (YY/YY) ---")
    season_pattern = re.compile(r'\d{2}/\d{2}')
    seasons = season_pattern.findall(career_soup.get_text())
    unique_seasons = sorted(set(seasons))
    print(f"Seasons found: {unique_seasons[:15]}")
    
    # Try to find rows with season + club
    print("\n--- First table preview ---")
    if tables:
        first_table = tables[0]
        rows = first_table.find_all('tr')[:5]
        for row in rows:
            cells = row.find_all(['td', 'th'])
            cell_text = [c.get_text(strip=True)[:20] for c in cells[:6]]
            print(f"  {cell_text}")

Fetching career stats: https://www.transfermarkt.com/lionel-messi/leistungsdatendetails/spieler/28003
  Waiting 3s before request...
  Fetching: https://www.transfermarkt.com/lionel-messi/leistungsdatendetails/spieler/28003
Saved to debug_messi_career.html

--- Looking for tables ---
Found 3 tables

--- Looking for club names ---
  'PSG': 0 matches
  'Barcelona': 2 matches
  'Inter Miami': 2 matches
  'Miami': 3 matches

--- Looking for season patterns (YY/YY) ---
Seasons found: ['03/04', '04/05', '05/06', '06/07', '07/08', '08/09', '09/10', '10/11', '11/12', '12/13', '13/14', '14/15', '15/07', '15/16', '16/17']

--- First table preview ---
  ['Filter by season:', 'All seasons25/2624/2', '']
  ['Filter by club:', 'All clubsInter Miami', '']
  ['League ranking / Lea', 'All typesFirst TierT', '']
  ['Filter by competitio', 'All competitionsCopa', '']
  ['Filter by position:', 'All positionsLeft Wi', '']


In [19]:
# Look for transfer links with aria-label containing "->"
if debug_soup:  # Using the transfers page we already fetched
    print("--- Looking for transfer links with aria-label ---")
    
    # Find all <a> tags with aria-label containing "->"
    transfer_links = debug_soup.find_all('a', attrs={'aria-label': re.compile(r'->')})
    print(f"Found {len(transfer_links)} links with '->' in aria-label")
    
    for link in transfer_links:
        aria = link.get('aria-label', '')
        href = link.get('href', '')
        print(f"  {aria}")
        print(f"    href: {href}\n")
    
    # Also try finding by the svelte class
    print("\n--- Looking for svelte links ---")
    svelte_links = debug_soup.find_all('a', class_=re.compile(r'svelte'))
    print(f"Found {len(svelte_links)} svelte links")
    
    for link in svelte_links[:10]:  # First 10
        aria = link.get('aria-label', '')
        title = link.get('title', '')
        href = link.get('href', '')
        if aria or title:
            print(f"  aria: {aria}")
            print(f"  title: {title}")
            print(f"  href: {href}\n")

--- Looking for transfer links with aria-label ---
Found 0 links with '->' in aria-label

--- Looking for svelte links ---
Found 0 svelte links


In [20]:
# Parse the career stats table more thoroughly
if career_soup:
    print("--- Parsing career stats tables ---\n")
    
    # Look for the main stats table (usually has class with 'responsive')
    tables = career_soup.find_all('table')
    
    for i, table in enumerate(tables):
        print(f"=== Table {i} ===")
        rows = table.find_all('tr')
        print(f"Rows: {len(rows)}")
        
        # Look for rows with season info
        for row in rows[:15]:  # First 15 rows
            cells = row.find_all(['td', 'th'])
            
            # Check for links to clubs
            club_links = row.find_all('a', href=re.compile(r'/verein/'))
            season_cell = row.find('td', class_=re.compile(r'zentriert'))
            
            # Get all text
            row_text = ' | '.join([c.get_text(strip=True)[:25] for c in cells[:8]])
            
            if club_links or (season_cell and re.search(r'\d{2}/\d{2}', row_text)):
                print(f"  {row_text}")
                for cl in club_links:
                    print(f"    -> Club link: {cl.get('href')} = {cl.get_text(strip=True)}")
        print()

--- Parsing career stats tables ---

=== Table 0 ===
Rows: 6

=== Table 1 ===
Rows: 93
  2026 |  | MLS |  | 2 | 2 | - | - / - / -
    -> Club link: /inter-miami-cf/startseite/verein/69261/saison_id/2025 = 
    -> Club link: /lionel-messi/leistungsdatendetails/spieler/28003/plus/0/saison/2025/wettbewerb/MLS1/verein/69261 = 2
  2025 |  | MLS Cup Playoffs |  | 6 | 6 | 7 | 1 / - / -
    -> Club link: /inter-miami-cf/startseite/verein/69261/saison_id/2024 = 
    -> Club link: /lionel-messi/leistungsdatendetails/spieler/28003/plus/0/saison/2024/wettbewerb/POUS/verein/69261 = 6
  2025 |  | MLS |  | 28 | 29 | 16 | 2 / - / -
    -> Club link: /inter-miami-cf/startseite/verein/69261/saison_id/2024 = 
    -> Club link: /lionel-messi/leistungsdatendetails/spieler/28003/plus/0/saison/2024/wettbewerb/MLS1/verein/69261 = 28
  2025 |  | Leagues Cup |  | 4 | 2 | 2 | 1 / - / -
    -> Club link: /inter-miami-cf/startseite/verein/69261/saison_id/2024 = 
    -> Club link: /lionel-messi/leistungsdatendetail

In [21]:
def get_club_for_season(player_id, season_year, player_slug="anyplayer"):
    """
    Get a player's club for a given season from their career stats page.
    
    Args:
        player_id: Transfermarkt player ID
        season_year: The season year (e.g., 2022 for the 22/23 season)
        player_slug: Player name slug for URL (optional, 'anyplayer' works)
    
    Returns:
        dict with club name, club_id, or None if not found
    """
    url = f"{BASE_URL}/{player_slug}/leistungsdatendetails/spieler/{player_id}"
    soup = fetch_page(url, delay=True)
    
    if not soup:
        return None
    
    # Build season pattern - look for both "YY/YY" format and single year
    season_short = f"{str(season_year)[2:]}/{str(season_year + 1)[2:]}"  # e.g., "22/23"
    
    clubs_found = {}
    
    # Find all rows with club links
    tables = soup.find_all('table')
    for table in tables:
        rows = table.find_all('tr')
        for row in rows:
            row_text = row.get_text()
            
            # Check if this row matches our season
            if season_short in row_text or str(season_year) in row_text:
                # Find club link in this row
                club_links = row.find_all('a', href=re.compile(r'/startseite/verein/(\d+)'))
                for link in club_links:
                    href = link.get('href', '')
                    match = re.search(r'/verein/(\d+)', href)
                    if match:
                        club_id = match.group(1)
                        # Extract club name from URL
                        club_name_match = re.search(r'/([^/]+)/startseite/verein/', href)
                        club_name = club_name_match.group(1).replace('-', ' ').title() if club_name_match else ''
                        
                        if club_id not in clubs_found:
                            clubs_found[club_id] = {
                                'club_id': club_id,
                                'club_name': club_name,
                                'href': href
                            }
    
    # Return the first club found (usually the main club for that season)
    if clubs_found:
        return list(clubs_found.values())[0]
    return None

# Test with Messi for 2022 World Cup (22/23 season)
print("Testing get_club_for_season for Messi (2022 WC)...")
result = get_club_for_season(28003, 2022)
print(f"Result: {result}")

Testing get_club_for_season for Messi (2022 WC)...
  Waiting 3s before request...
  Fetching: https://www.transfermarkt.com/anyplayer/leistungsdatendetails/spieler/28003
Result: {'club_id': '583', 'club_name': 'Fc Paris Saint Germain', 'href': '/fc-paris-saint-germain/startseite/verein/583/saison_id/2022'}


In [22]:
# MUCH BETTER APPROACH: Scrape the lineup page directly!
# URL format: /team1_team2/aufstellung/spielbericht/{match_id}

def scrape_lineup_page(match_url):
    """
    Scrape a Transfermarkt match lineup page to get both teams' starting 11s
    with their clubs at the time of the match.
    
    Returns dict with home_team and away_team lineups.
    """
    soup = fetch_page(match_url, delay=True)
    if not soup:
        return None
    
    result = {
        'match_url': match_url,
        'home_team': {'name': '', 'players': []},
        'away_team': {'name': '', 'players': []}
    }
    
    # Find the two "Starting Line-up" sections
    lineup_headers = soup.find_all(['h2', 'div'], string=re.compile(r'Starting Line-up', re.I))
    print(f"Found {len(lineup_headers)} lineup headers")
    
    # Alternative: Find tables with player data
    # Look for links with /profil/spieler/ pattern
    player_links = soup.find_all('a', href=re.compile(r'/profil/spieler/\d+'))
    print(f"Found {len(player_links)} player profile links")
    
    # Get unique players (each player appears multiple times)
    seen_players = set()
    all_players = []
    
    for link in player_links:
        href = link.get('href', '')
        player_match = re.search(r'/([^/]+)/profil/spieler/(\d+)', href)
        if player_match:
            slug = player_match.group(1)
            player_id = player_match.group(2)
            
            if player_id not in seen_players:
                seen_players.add(player_id)
                
                # Get player name from link text or title
                name = link.get('title', '') or link.get_text(strip=True)
                
                # Find the parent row to get club info
                parent_row = link.find_parent('tr')
                club_info = None
                jersey_num = None
                
                if parent_row:
                    # Look for club link in same row
                    club_link = parent_row.find('a', href=re.compile(r'/startseite/verein/\d+'))
                    if club_link:
                        club_href = club_link.get('href', '')
                        club_match = re.search(r'/([^/]+)/startseite/verein/(\d+)', club_href)
                        if club_match:
                            club_info = {
                                'slug': club_match.group(1),
                                'club_id': club_match.group(2),
                                'name': club_link.get('title', '') or club_match.group(1).replace('-', ' ').title()
                            }
                    
                    # Get jersey number (usually first td)
                    first_td = parent_row.find('td')
                    if first_td:
                        num_text = first_td.get_text(strip=True)
                        if num_text.isdigit():
                            jersey_num = int(num_text)
                
                if name and club_info:  # Only add if we have both
                    all_players.append({
                        'name': name,
                        'player_id': player_id,
                        'slug': slug,
                        'jersey': jersey_num,
                        'club': club_info
                    })
    
    print(f"Extracted {len(all_players)} unique players with club info")
    return all_players

# Test with the 2022 World Cup Final
test_url = "https://www.transfermarkt.com/argentina_france/aufstellung/spielbericht/3975879"
print(f"Scraping: {test_url}\n")
players = scrape_lineup_page(test_url)

if players:
    print("\n--- First 15 players found ---")
    for p in players[:15]:
        print(f"  #{p.get('jersey', '?')} {p['name']} - {p['club']['name']} (ID: {p['club']['club_id']})")

Scraping: https://www.transfermarkt.com/argentina_france/aufstellung/spielbericht/3975879

  Waiting 3s before request...
  Fetching: https://www.transfermarkt.com/argentina_france/aufstellung/spielbericht/3975879
Found 0 lineup headers
Found 50 player profile links
Extracted 0 unique players with club info


In [23]:
# Debug: Inspect the lineup page HTML structure
lineup_url = "https://www.transfermarkt.com/argentina_france/aufstellung/spielbericht/3975879"
lineup_soup = fetch_page(lineup_url, delay=True)

if lineup_soup:
    # Save for inspection
    with open('debug_lineup_page.html', 'w', encoding='utf-8') as f:
        f.write(lineup_soup.prettify())
    print("Saved to debug_lineup_page.html")
    
    # Find a specific player link and examine its context
    messi_link = lineup_soup.find('a', href=re.compile(r'lionel-messi.*profil'))
    if messi_link:
        print("\n--- Found Messi link ---")
        print(f"Text: {messi_link.get_text(strip=True)}")
        print(f"Href: {messi_link.get('href')}")
        
        # Look at parent elements
        print("\n--- Parent chain ---")
        parent = messi_link.parent
        for i in range(5):
            if parent:
                print(f"Level {i}: <{parent.name}> class={parent.get('class')}")
                parent = parent.parent
        
        # Look for club link nearby
        print("\n--- Looking for club link near Messi ---")
        # Go up to find a container that has both player and club
        container = messi_link.find_parent('tr') or messi_link.find_parent('div', class_=re.compile(r'row|item|player'))
        if container:
            print(f"Container: <{container.name}> class={container.get('class')}")
            club_links = container.find_all('a', href=re.compile(r'verein/\d+'))
            print(f"Club links in container: {len(club_links)}")
            for cl in club_links:
                print(f"  - {cl.get('href')} | title={cl.get('title')}")
        
        # Try finding club link as sibling
        print("\n--- All links near Messi (siblings + parent siblings) ---")
        parent_container = messi_link.find_parent(['tr', 'div', 'td'])
        if parent_container:
            all_links = parent_container.find_all('a')[:10]
            for link in all_links:
                print(f"  {link.get('href', '')[:60]}... | {link.get_text(strip=True)[:30]}")
    else:
        print("Messi link not found!")
    
    # Check for tables
    print(f"\n--- Tables on page: {len(lineup_soup.find_all('table'))} ---")
    
    # Look for the lineup section
    print("\n--- Looking for lineup-related divs ---")
    lineup_divs = lineup_soup.find_all(['div', 'section'], class_=re.compile(r'lineup|aufstellung|starting', re.I))
    print(f"Found {len(lineup_divs)} lineup-related divs")

  Waiting 3s before request...
  Fetching: https://www.transfermarkt.com/argentina_france/aufstellung/spielbericht/3975879
Saved to debug_lineup_page.html

--- Found Messi link ---
Text: 
Href: /lionel-messi/profil/spieler/28003

--- Parent chain ---
Level 0: <td> class=None
Level 1: <tr> class=None
Level 2: <table> class=['inline-table']
Level 3: <td> class=[]
Level 4: <tr> class=None

--- Looking for club link near Messi ---
Container: <tr> class=None
Club links in container: 0

--- All links near Messi (siblings + parent siblings) ---
  /lionel-messi/profil/spieler/28003... | 

--- Tables on page: 63 ---

--- Looking for lineup-related divs ---
Found 0 lineup-related divs


In [24]:
# Debug: Look at the OUTER row structure (parent of the inline-table)
if lineup_soup:
    messi_link = lineup_soup.find('a', href=re.compile(r'lionel-messi.*profil'))
    if messi_link:
        # Navigate up: td -> tr -> table.inline-table -> td -> tr (outer row)
        inline_table = messi_link.find_parent('table', class_='inline-table')
        if inline_table:
            outer_td = inline_table.find_parent('td')
            outer_row = outer_td.find_parent('tr') if outer_td else None
            
            if outer_row:
                print("--- Outer row contents ---")
                cells = outer_row.find_all('td', recursive=False)
                print(f"Found {len(cells)} cells in outer row\n")
                
                for i, cell in enumerate(cells):
                    print(f"Cell {i}:")
                    # Get text preview
                    text = cell.get_text(strip=True)[:50]
                    print(f"  Text: {text}")
                    
                    # Get all links
                    links = cell.find_all('a')
                    for link in links[:3]:
                        href = link.get('href', '')[:60]
                        title = link.get('title', '')[:30]
                        print(f"  Link: {href}... | title={title}")
                    print()
                
                # Look specifically for club/verein links in outer row
                print("--- Club links in outer row ---")
                club_links = outer_row.find_all('a', href=re.compile(r'/startseite/verein/\d+'))
                for cl in club_links:
                    print(f"  {cl.get('href')}")
                    print(f"  title: {cl.get('title')}")

--- Outer row contents ---
Found 4 cells in outer row

Cell 0:
  Text: 10

Cell 1:
  Text: Lionel Messi(35 years old)Right Winger, €50.00m
  Link: /lionel-messi/profil/spieler/28003... | title=
  Link: /lionel-messi/leistungsdatendetails/spieler/28003/saison/202... | title=Lionel Messi

Cell 2:
  Text: 

Cell 3:
  Text: 
  Link: /fc-paris-saint-germain/startseite/verein/583/saison_id/2021... | title=Paris Saint-Germain

--- Club links in outer row ---
  /fc-paris-saint-germain/startseite/verein/583/saison_id/2021
  title: Paris Saint-Germain


In [25]:
def scrape_match_lineup(match_url):
    """
    Scrape a Transfermarkt match lineup page.
    Returns dict with both teams' starting 11s and subs.
    """
    soup = fetch_page(match_url, delay=True)
    if not soup:
        return None
    
    players = []
    
    # Find all rows that have an inline-table (player rows)
    inline_tables = soup.find_all('table', class_='inline-table')
    
    for inline_table in inline_tables:
        # Navigate up to outer row
        outer_td = inline_table.find_parent('td')
        if not outer_td:
            continue
        outer_row = outer_td.find_parent('tr')
        if not outer_row:
            continue
            
        cells = outer_row.find_all('td', recursive=False)
        if len(cells) < 4:
            continue
        
        # Cell 0: Jersey number
        jersey_text = cells[0].get_text(strip=True)
        jersey = int(jersey_text) if jersey_text.isdigit() else None
        
        # Cell 1: Player info
        player_link = cells[1].find('a', href=re.compile(r'/profil/spieler/\d+'))
        if not player_link:
            continue
            
        href = player_link.get('href', '')
        player_match = re.search(r'/([^/]+)/profil/spieler/(\d+)', href)
        if not player_match:
            continue
            
        player_slug = player_match.group(1)
        player_id = player_match.group(2)
        
        # Get player name - try title attr first, then the text link nearby
        name_link = cells[1].find('a', href=re.compile(r'/leistungsdatendetails/'))
        player_name = name_link.get('title', '') if name_link else player_slug.replace('-', ' ').title()
        
        # Cell 3: Club info
        club_link = cells[3].find('a', href=re.compile(r'/startseite/verein/\d+'))
        if not club_link:
            continue
            
        club_href = club_link.get('href', '')
        club_match = re.search(r'/([^/]+)/startseite/verein/(\d+)', club_href)
        if not club_match:
            continue
            
        club_slug = club_match.group(1)
        club_id = club_match.group(2)
        club_name = club_link.get('title', '') or club_slug.replace('-', ' ').title()
        
        players.append({
            'jersey': jersey,
            'name': player_name,
            'player_id': player_id,
            'player_slug': player_slug,
            'club_name': club_name,
            'club_id': club_id,
            'club_slug': club_slug
        })
    
    return players

# Test it!
print("Scraping 2022 World Cup Final lineup...\n")
match_url = "https://www.transfermarkt.com/argentina_france/aufstellung/spielbericht/3975879"
players = scrape_match_lineup(match_url)

if players:
    print(f"Found {len(players)} players!\n")
    print("--- Argentina (first 11 with club) ---")
    for p in players[:11]:
        print(f"  #{p['jersey']:2} {p['name']:<25} - {p['club_name']}")
    print(f"\n--- France (next 11) ---")
    for p in players[11:22]:
        print(f"  #{p['jersey']:2} {p['name']:<25} - {p['club_name']}")

Scraping 2022 World Cup Final lineup...

  Waiting 3s before request...
  Fetching: https://www.transfermarkt.com/argentina_france/aufstellung/spielbericht/3975879
Found 50 players!

--- Argentina (first 11 with club) ---
  #23 Emiliano Martínez         - Aston Villa
  #13 Cristian Romero           - Tottenham Hotspur
  #19 Nicolás Otamendi          - SL Benfica
  # 3 Nicolás Tagliafico        - Olympique Lyon
  #26 Nahuel Molina             - Atlético de Madrid
  #24 Enzo Fernández            - SL Benfica
  # 7 Rodrigo De Paul           - Atlético de Madrid
  #20 Alexis Mac Allister       - Brighton & Hove Albion
  #11 Ángel Di María            - Juventus FC
  #10 Lionel Messi              - Paris Saint-Germain
  # 9 Julián Alvarez            - Manchester City

--- France (next 11) ---
  # 1 Hugo Lloris               - Tottenham Hotspur
  # 4 Raphaël Varane            - Manchester United
  #18 Dayot Upamecano           - Bayern Munich
  #22 Theo Hernández            - AC Milan
  # 5 J

In [26]:
# Generate a Starting11 quiz from the scraped data
def generate_starting11_quiz(players, team_name, country_code, match_info):
    """
    Generate a quiz JSON for Starting11 mode.
    
    Args:
        players: List of 11 player dicts (starting lineup only)
        team_name: e.g., "Argentina"
        country_code: e.g., "ARG" 
        match_info: dict with tournament, date, opponent, etc.
    """
    quiz = {
        "answer": {
            "country": team_name,
            "country_code": country_code
        },
        "match": match_info,
        "players": []
    }
    
    for p in players[:11]:  # Only starting 11
        quiz["players"].append({
            "jersey": p["jersey"],
            "name": p["name"],
            "player_id": p["player_id"],
            "club": {
                "name": p["club_name"],
                "club_id": p["club_id"],
                "badge_url": f"https://tmssl.akamaized.net/images/wappen/head/{p['club_id']}.png"
            }
        })
    
    return quiz

# Generate quiz for Argentina 2022 World Cup Final
argentina_quiz = generate_starting11_quiz(
    players[:11],  # First 11 are Argentina
    team_name="Argentina",
    country_code="ARG",
    match_info={
        "tournament": "World Cup 2022",
        "stage": "Final",
        "date": "2022-12-18",
        "opponent": "France",
        "result": "3-3 (4-2 pens)",
        "venue": "Lusail Stadium"
    }
)

# Generate quiz for France too
france_quiz = generate_starting11_quiz(
    players[11:22],  # Next 11 are France
    team_name="France",
    country_code="FRA",
    match_info={
        "tournament": "World Cup 2022",
        "stage": "Final", 
        "date": "2022-12-18",
        "opponent": "Argentina",
        "result": "3-3 (2-4 pens)",
        "venue": "Lusail Stadium"
    }
)

print("Argentina Quiz Preview:")
print(f"  Answer: {argentina_quiz['answer']['country']}")
print(f"  Match: {argentina_quiz['match']['tournament']} {argentina_quiz['match']['stage']}")
print(f"  Players: {len(argentina_quiz['players'])}")
for p in argentina_quiz['players'][:3]:
    print(f"    #{p['jersey']} {p['name']} - {p['club']['name']}")
print("    ...")

print(f"\nFrance Quiz Preview:")
print(f"  Answer: {france_quiz['answer']['country']}")
print(f"  Players: {len(france_quiz['players'])}")
for p in france_quiz['players'][:3]:
    print(f"    #{p['jersey']} {p['name']} - {p['club']['name']}")
print("    ...")

Argentina Quiz Preview:
  Answer: Argentina
  Match: World Cup 2022 Final
  Players: 11
    #23 Emiliano Martínez - Aston Villa
    #13 Cristian Romero - Tottenham Hotspur
    #19 Nicolás Otamendi - SL Benfica
    ...

France Quiz Preview:
  Answer: France
  Players: 11
    #1 Hugo Lloris - Tottenham Hotspur
    #4 Raphaël Varane - Manchester United
    #18 Dayot Upamecano - Bayern Munich
    ...


In [27]:
# Save quizzes and create visualization
import json
import os

# Create output directory
os.makedirs('../quizzes/starting11/wc2022', exist_ok=True)

# Save Argentina quiz
with open('../quizzes/starting11/wc2022/final_argentina.json', 'w') as f:
    json.dump(argentina_quiz, f, indent=2)
print("Saved: quizzes/starting11/wc2022/final_argentina.json")

# Save France quiz  
with open('../quizzes/starting11/wc2022/final_france.json', 'w') as f:
    json.dump(france_quiz, f, indent=2)
print("Saved: quizzes/starting11/wc2022/final_france.json")

# Create HTML visualization with real data
html = '''<!DOCTYPE html>
<html>
<head>
    <title>Starting11 - 2022 World Cup Final</title>
    <style>
        * { margin: 0; padding: 0; box-sizing: border-box; }
        body { font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; background: #1a1a2e; color: white; }
        .container { max-width: 900px; margin: 0 auto; padding: 20px; }
        h1 { text-align: center; margin-bottom: 10px; }
        .subtitle { text-align: center; color: #888; margin-bottom: 30px; }
        
        .pitch {
            position: relative;
            width: 100%;
            aspect-ratio: 1.5;
            background: linear-gradient(to bottom, #2d5a27, #3d7a37);
            border-radius: 8px;
            border: 3px solid white;
            overflow: hidden;
        }
        
        .pitch::before {
            content: '';
            position: absolute;
            top: 50%;
            left: 0;
            right: 0;
            height: 2px;
            background: rgba(255,255,255,0.5);
        }
        
        .pitch::after {
            content: '';
            position: absolute;
            top: 50%;
            left: 50%;
            transform: translate(-50%, -50%);
            width: 120px;
            height: 120px;
            border: 2px solid rgba(255,255,255,0.5);
            border-radius: 50%;
        }
        
        .player {
            position: absolute;
            display: flex;
            flex-direction: column;
            align-items: center;
            transform: translate(-50%, -50%);
        }
        
        .badge {
            width: 50px;
            height: 50px;
            background: white;
            border-radius: 8px;
            padding: 4px;
            box-shadow: 0 2px 8px rgba(0,0,0,0.3);
        }
        
        .badge img {
            width: 100%;
            height: 100%;
            object-fit: contain;
        }
        
        .jersey {
            margin-top: 4px;
            background: rgba(0,0,0,0.7);
            padding: 2px 8px;
            border-radius: 10px;
            font-size: 12px;
            font-weight: bold;
        }
        
        .answer-box {
            margin-top: 30px;
            text-align: center;
        }
        
        .answer-box input {
            padding: 12px 20px;
            font-size: 18px;
            border: none;
            border-radius: 8px;
            width: 300px;
        }
        
        .answer-box button {
            padding: 12px 30px;
            font-size: 18px;
            background: #4CAF50;
            color: white;
            border: none;
            border-radius: 8px;
            cursor: pointer;
            margin-left: 10px;
        }
    </style>
</head>
<body>
    <div class="container">
        <h1>🏆 Starting 11</h1>
        <p class="subtitle">World Cup 2022 Final • Guess the National Team</p>
        
        <div class="pitch">
'''

# 4-3-3 formation positions (percentage based)
positions_433 = [
    (50, 90),   # GK
    (20, 70),   # LB
    (40, 72),   # CB
    (60, 72),   # CB
    (80, 70),   # RB
    (25, 50),   # CM
    (50, 45),   # CM
    (75, 50),   # CM
    (20, 25),   # LW
    (50, 20),   # ST
    (80, 25),   # RW
]

# Add Argentina players with real badge URLs
for i, p in enumerate(argentina_quiz['players']):
    x, y = positions_433[i]
    badge_url = p['club']['badge_url']
    html += f'''
            <div class="player" style="left: {x}%; top: {y}%;">
                <div class="badge"><img src="{badge_url}" alt="{p['club']['name']}"></div>
                <span class="jersey">#{p['jersey']}</span>
            </div>
'''

html += '''
        </div>
        
        <div class="answer-box">
            <input type="text" placeholder="Which national team is this?" id="guess">
            <button onclick="checkAnswer()">Submit</button>
        </div>
    </div>
    
    <script>
        function checkAnswer() {
            const guess = document.getElementById('guess').value.toLowerCase();
            if (guess.includes('argentin')) {
                alert('🎉 Correct! This is Argentina\\'s starting 11 from the 2022 World Cup Final!');
            } else {
                alert('❌ Try again!');
            }
        }
    </script>
</body>
</html>'''

with open('starting11_real_data.html', 'w') as f:
    f.write(html)
print("\nSaved: starting11_real_data.html")
print("\n✅ Open this file in a browser to see the visualization with real club badges!")

Saved: quizzes/starting11/wc2022/final_argentina.json
Saved: quizzes/starting11/wc2022/final_france.json

Saved: starting11_real_data.html

✅ Open this file in a browser to see the visualization with real club badges!


In [ ]:
# Scrape the match INDEX page to get formation and player positions
def scrape_match_formations(match_id):
    """
    Scrape the match index page to get formation info.
    URL: /team1_team2/index/spielbericht/{match_id}
    """
    url = f"{BASE_URL}/argentina_france/index/spielbericht/{match_id}"
    soup = fetch_page(url, delay=True)
    
    if not soup:
        return None
    
    # Save for debugging
    with open('debug_match_index.html', 'w', encoding='utf-8') as f:
        f.write(soup.prettify())
    print("Saved: debug_match_index.html")
    
    # Look for formation info - usually in text like "4-3-3" or in a class
    formations = {}
    
    # Find "Starting Line-up: X-X-X" text
    lineup_texts = soup.find_all(string=re.compile(r'Starting Line-up.*\d-\d-\d'))
    print(f"Found {len(lineup_texts)} lineup texts")
    for lt in lineup_texts:
        print(f"  {lt.strip()}")
    
    # Look for the tactical lineup visualization
    # These usually have player positions in divs with specific coordinates/classes
    print("\n--- Looking for lineup visualization ---")
    
    # Find divs with class containing 'aufstellung' (German for lineup) or 'lineup'
    lineup_divs = soup.find_all(['div', 'ul'], class_=re.compile(r'aufstellung|lineup|formation|tactics', re.I))
    print(f"Found {len(lineup_divs)} lineup-related elements")
    
    # Look for player entries with position info
    # The visual shows jersey numbers at specific positions
    print("\n--- Looking for positioned player elements ---")
    
    # Find elements with style attributes containing position (top/left)
    positioned = soup.find_all(style=re.compile(r'(top|left):\s*\d+'))
    print(f"Found {len(positioned)} elements with position styles")
    
    # Look for the specific lineup structure - often in <li> tags
    lineup_items = soup.find_all('li', class_=re.compile(r'[A-Z]{2}'))  # Like "GK", "CB", etc.
    print(f"Found {len(lineup_items)} lineup position items")
    
    return soup

# Test
print("Scraping match index page for formations...\n")
match_soup = scrape_match_formations(3975879)

In [ ]:
# Define formation position templates (x%, y% on pitch - 0,0 is top-left, defending at bottom)
FORMATIONS = {
    "4-3-3": {
        "positions": [
            {"role": "GK", "x": 50, "y": 92},
            {"role": "LB", "x": 15, "y": 72},
            {"role": "CB", "x": 35, "y": 75},
            {"role": "CB", "x": 65, "y": 75},
            {"role": "RB", "x": 85, "y": 72},
            {"role": "CM", "x": 30, "y": 52},
            {"role": "CM", "x": 50, "y": 48},
            {"role": "CM", "x": 70, "y": 52},
            {"role": "LW", "x": 18, "y": 25},
            {"role": "ST", "x": 50, "y": 18},
            {"role": "RW", "x": 82, "y": 25},
        ]
    },
    "4-2-3-1": {
        "positions": [
            {"role": "GK", "x": 50, "y": 92},
            {"role": "LB", "x": 15, "y": 72},
            {"role": "CB", "x": 35, "y": 75},
            {"role": "CB", "x": 65, "y": 75},
            {"role": "RB", "x": 85, "y": 72},
            {"role": "CDM", "x": 35, "y": 55},
            {"role": "CDM", "x": 65, "y": 55},
            {"role": "CAM", "x": 50, "y": 38},
            {"role": "LW", "x": 20, "y": 30},
            {"role": "ST", "x": 50, "y": 15},
            {"role": "RW", "x": 80, "y": 30},
        ]
    }
}

# Argentina's actual lineup order from Transfermarkt (4-3-3 Attacking)
# Based on the visual: GK, then back 4 (L to R), then mid 3, then front 3
ARGENTINA_2022_FINAL = {
    "formation": "4-3-3",
    "player_order": [
        {"jersey": 23, "role": "GK"},      # Martínez
        {"jersey": 3, "role": "LB"},       # Tagliafico
        {"jersey": 19, "role": "CB"},      # Otamendi
        {"jersey": 13, "role": "CB"},      # Romero
        {"jersey": 26, "role": "RB"},      # Molina
        {"jersey": 24, "role": "CM"},      # Enzo Fernández
        {"jersey": 7, "role": "CM"},       # De Paul
        {"jersey": 20, "role": "CM"},      # Mac Allister
        {"jersey": 11, "role": "LW"},      # Di María
        {"jersey": 9, "role": "ST"},       # Álvarez
        {"jersey": 10, "role": "RW"},      # Messi
    ]
}

FRANCE_2022_FINAL = {
    "formation": "4-2-3-1",
    "player_order": [
        {"jersey": 1, "role": "GK"},       # Lloris
        {"jersey": 22, "role": "LB"},      # Theo Hernández
        {"jersey": 18, "role": "CB"},      # Upamecano
        {"jersey": 4, "role": "CB"},       # Varane
        {"jersey": 5, "role": "RB"},       # Koundé
        {"jersey": 8, "role": "CDM"},      # Tchouaméni
        {"jersey": 14, "role": "CDM"},     # Rabiot
        {"jersey": 7, "role": "CAM"},      # Griezmann
        {"jersey": 11, "role": "LW"},      # Dembélé
        {"jersey": 9, "role": "ST"},       # Giroud
        {"jersey": 10, "role": "RW"},      # Mbappé
    ]
}

print("Formation templates defined!")
print(f"Argentina: {ARGENTINA_2022_FINAL['formation']}")
print(f"France: {FRANCE_2022_FINAL['formation']}")

In [ ]:
# Update quiz data with proper positions
def add_positions_to_quiz(quiz, formation_info, formation_template):
    """Add x, y positions to each player based on formation."""
    positions = formation_template["positions"]
    player_order = formation_info["player_order"]
    
    # Create jersey -> position mapping
    jersey_to_pos = {p["jersey"]: i for i, p in enumerate(player_order)}
    
    for player in quiz["players"]:
        jersey = player["jersey"]
        if jersey in jersey_to_pos:
            idx = jersey_to_pos[jersey]
            pos = positions[idx]
            player["position"] = {
                "role": pos["role"],
                "x": pos["x"],
                "y": pos["y"]
            }
    
    quiz["formation"] = formation_info["formation"]
    return quiz

# Update Argentina quiz
argentina_quiz = add_positions_to_quiz(
    argentina_quiz, 
    ARGENTINA_2022_FINAL, 
    FORMATIONS["4-3-3"]
)

# Update France quiz
france_quiz = add_positions_to_quiz(
    france_quiz,
    FRANCE_2022_FINAL,
    FORMATIONS["4-2-3-1"]
)

# Preview
print("Argentina positions:")
for p in argentina_quiz["players"]:
    pos = p.get("position", {})
    print(f"  #{p['jersey']:2} {p['name']:<22} - {pos.get('role', '?'):>3} ({pos.get('x', '?'):>2}, {pos.get('y', '?'):>2})")

# Save updated quizzes
with open('../quizzes/starting11/wc2022/final_argentina.json', 'w') as f:
    json.dump(argentina_quiz, f, indent=2)
    
with open('../quizzes/starting11/wc2022/final_france.json', 'w') as f:
    json.dump(france_quiz, f, indent=2)
    
print("\n✅ Quizzes updated with positions!")

In [ ]:
# Regenerate HTML visualization with correct positions
html = '''<!DOCTYPE html>
<html>
<head>
    <title>Starting11 - 2022 World Cup Final</title>
    <style>
        * { margin: 0; padding: 0; box-sizing: border-box; }
        body { font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; background: #1a1a2e; color: white; }
        .container { max-width: 900px; margin: 0 auto; padding: 20px; }
        h1 { text-align: center; margin-bottom: 10px; }
        .subtitle { text-align: center; color: #888; margin-bottom: 30px; }
        
        .pitch {
            position: relative;
            width: 100%;
            aspect-ratio: 1.4;
            background: linear-gradient(to bottom, #2d5a27 0%, #3d7a37 50%, #2d5a27 100%);
            border-radius: 8px;
            border: 3px solid rgba(255,255,255,0.8);
            overflow: hidden;
        }
        
        /* Center circle */
        .pitch::before {
            content: '';
            position: absolute;
            top: 50%;
            left: 50%;
            transform: translate(-50%, -50%);
            width: 100px;
            height: 100px;
            border: 2px solid rgba(255,255,255,0.4);
            border-radius: 50%;
        }
        
        /* Halfway line */
        .pitch::after {
            content: '';
            position: absolute;
            top: 50%;
            left: 0;
            right: 0;
            height: 2px;
            background: rgba(255,255,255,0.4);
        }
        
        /* Penalty areas */
        .penalty-area-top, .penalty-area-bottom {
            position: absolute;
            left: 50%;
            transform: translateX(-50%);
            width: 44%;
            height: 16%;
            border: 2px solid rgba(255,255,255,0.4);
        }
        .penalty-area-top { top: 0; border-top: none; }
        .penalty-area-bottom { bottom: 0; border-bottom: none; }
        
        /* Goal areas */
        .goal-area-top, .goal-area-bottom {
            position: absolute;
            left: 50%;
            transform: translateX(-50%);
            width: 18%;
            height: 6%;
            border: 2px solid rgba(255,255,255,0.4);
        }
        .goal-area-top { top: 0; border-top: none; }
        .goal-area-bottom { bottom: 0; border-bottom: none; }
        
        .player {
            position: absolute;
            display: flex;
            flex-direction: column;
            align-items: center;
            transform: translate(-50%, -50%);
            transition: transform 0.2s;
        }
        
        .player:hover {
            transform: translate(-50%, -50%) scale(1.1);
            z-index: 10;
        }
        
        .badge {
            width: 48px;
            height: 48px;
            background: white;
            border-radius: 8px;
            padding: 3px;
            box-shadow: 0 2px 8px rgba(0,0,0,0.4);
        }
        
        .badge img {
            width: 100%;
            height: 100%;
            object-fit: contain;
        }
        
        .jersey {
            margin-top: 4px;
            background: rgba(0,0,0,0.8);
            padding: 2px 8px;
            border-radius: 10px;
            font-size: 11px;
            font-weight: bold;
        }
        
        .player-name {
            font-size: 9px;
            color: rgba(255,255,255,0.7);
            margin-top: 2px;
            text-shadow: 0 1px 2px rgba(0,0,0,0.8);
            display: none;
        }
        
        .player:hover .player-name {
            display: block;
        }
        
        .answer-box {
            margin-top: 30px;
            text-align: center;
        }
        
        .answer-box input {
            padding: 12px 20px;
            font-size: 18px;
            border: none;
            border-radius: 8px;
            width: 300px;
        }
        
        .answer-box button {
            padding: 12px 30px;
            font-size: 18px;
            background: #4CAF50;
            color: white;
            border: none;
            border-radius: 8px;
            cursor: pointer;
            margin-left: 10px;
        }
        
        .answer-box button:hover {
            background: #45a049;
        }
        
        .formation-label {
            position: absolute;
            bottom: 10px;
            right: 10px;
            background: rgba(0,0,0,0.6);
            padding: 4px 10px;
            border-radius: 4px;
            font-size: 12px;
        }
    </style>
</head>
<body>
    <div class="container">
        <h1>⚽ Starting 11</h1>
        <p class="subtitle">World Cup 2022 Final • Guess the National Team</p>
        
        <div class="pitch">
            <div class="penalty-area-top"></div>
            <div class="goal-area-top"></div>
            <div class="penalty-area-bottom"></div>
            <div class="goal-area-bottom"></div>
'''

# Add players at their actual positions
for p in argentina_quiz['players']:
    pos = p.get('position', {})
    x = pos.get('x', 50)
    y = pos.get('y', 50)
    badge_url = p['club']['badge_url']
    html += f'''
            <div class="player" style="left: {x}%; top: {y}%;">
                <div class="badge"><img src="{badge_url}" alt="{p['club']['name']}" title="{p['club']['name']}"></div>
                <span class="jersey">#{p['jersey']}</span>
                <span class="player-name">{p['name']}</span>
            </div>
'''

html += f'''
            <div class="formation-label">{argentina_quiz.get('formation', '4-3-3')}</div>
        </div>
        
        <div class="answer-box">
            <input type="text" placeholder="Which national team is this?" id="guess">
            <button onclick="checkAnswer()">Submit</button>
        </div>
    </div>
    
    <script>
        function checkAnswer() {{
            const guess = document.getElementById('guess').value.toLowerCase();
            if (guess.includes('argentin')) {{
                alert('🎉 Correct! This is Argentina\\'s starting 11 from the 2022 World Cup Final vs France!');
            }} else {{
                alert('❌ Try again! Hint: They won the tournament.');
            }}
        }}
        
        document.getElementById('guess').addEventListener('keypress', function(e) {{
            if (e.key === 'Enter') checkAnswer();
        }});
    </script>
</body>
</html>'''

with open('starting11_final.html', 'w') as f:
    f.write(html)

print("✅ Saved: starting11_final.html")
print("\nOpen in browser to see Argentina's 4-3-3 formation with correct positions!")

In [ ]:
def extract_players_robust(soup):
    """
    More robust player extraction by looking at the actual page structure.
    """
    players_data = []
    
    # Find player links with their profile URLs
    # Pattern: /player-name/profil/spieler/12345
    player_links = soup.find_all('a', href=re.compile(r'/[\w-]+/profil/spieler/\d+'))
    
    seen_ids = set()
    for link in player_links:
        href = link.get('href', '')
        # Extract player ID
        match = re.search(r'/profil/spieler/(\d+)', href)
        if match:
            player_id = match.group(1)
            if player_id not in seen_ids:
                seen_ids.add(player_id)
                name = link.text.strip()
                if name and len(name) > 1:  # Filter out empty/short names
                    players_data.append({
                        'name': name,
                        'player_id': player_id,
                        'profile_url': BASE_URL + href if not href.startswith('http') else href
                    })
    
    return players_data

if soup:
    all_players = extract_players_robust(soup)
    print(f"Found {len(all_players)} unique players on page:\n")
    for i, p in enumerate(all_players[:25]):
        print(f"{i+1:2}. {p['name']:<25} (ID: {p['player_id']})")

## Step 3: Get Player's Club at Match Date

For each player, we need to scrape their profile to find which club they were at on the match date.

In [7]:
def get_player_club_at_date(player_id, match_date, player_name=""):
    """
    Scrape a player's profile to find which club they were at on match_date.
    
    Args:
        player_id: Transfermarkt player ID
        match_date: Date string in format 'YYYY-MM-DD'
        player_name: For logging purposes
    
    Returns:
        dict with club name and club_id, or None if not found
    """
    # Profile URL
    profile_url = f"{BASE_URL}/anyplayer/profil/spieler/{player_id}"
    
    print(f"  Getting club for {player_name} (ID: {player_id})...")
    
    soup = fetch_page(profile_url)
    if not soup:
        return None
    
    # Look for current club info on the profile
    # The profile page shows current club prominently
    # For historical data, we might need to check transfer history
    
    # Try to find current club from the info table
    info_table = soup.find('div', class_='info-table')
    
    # Look for club links in the player data section
    club_link = soup.find('a', href=re.compile(r'/[\w-]+/startseite/verein/\d+'))
    if club_link:
        club_name = club_link.text.strip()
        club_href = club_link.get('href', '')
        club_match = re.search(r'/verein/(\d+)', club_href)
        club_id = club_match.group(1) if club_match else None
        
        return {
            'club': club_name,
            'club_id': club_id
        }
    
    return None

# Test with one player (Messi)
# Messi's Transfermarkt ID: 28003
test_result = get_player_club_at_date('28003', '2022-12-18', 'Lionel Messi')
print(f"\nResult: {test_result}")

  Getting club for Lionel Messi (ID: 28003)...
  Waiting 3s before request...
  Fetching: https://www.transfermarkt.com/anyplayer/profil/spieler/28003

Result: {'club': '', 'club_id': '69261'}


In [ ]:
def get_player_transfer_history(player_id, player_name=""):
    """
    Get a player's transfer history to find club at any date.
    Uses the transfers page which has complete history.
    """
    transfers_url = f"{BASE_URL}/anyplayer/leistungsdatendetails/spieler/{player_id}"
    
    print(f"  Getting transfer history for {player_name}...")
    
    soup = fetch_page(transfers_url)
    if not soup:
        return []
    
    # Parse transfer/career entries
    transfers = []
    
    # Look for the career stats table
    # This shows which clubs they played for in which seasons
    rows = soup.find_all('tr')
    
    for row in rows:
        cells = row.find_all('td')
        if len(cells) >= 3:
            # Look for club links in this row
            club_link = row.find('a', href=re.compile(r'/verein/\d+'))
            season_cell = cells[0].text.strip() if cells else ''
            
            if club_link and re.match(r'\d{2}/\d{2}', season_cell):
                transfers.append({
                    'season': season_cell,
                    'club': club_link.text.strip()
                })
    
    return transfers

# Test with Messi
# history = get_player_transfer_history('28003', 'Lionel Messi')
# print(f"\nFound {len(history)} career entries")
# for h in history[:10]:
#     print(f"  {h['season']}: {h['club']}")

## Step 4: Put It All Together

Now let's build the complete scraper that:
1. Gets the starting 11 from a match
2. Gets each player's club
3. Outputs quiz JSON

In [8]:
# For testing, let's manually define the Argentina 2022 WC Final lineup
# We'll use this to test the club lookup and visualization

ARGENTINA_2022_FINAL = [
    {'name': 'Emiliano Martínez', 'player_id': '111873', 'position': 'GK', 'grid': (5, 1)},
    {'name': 'Nahuel Molina', 'player_id': '424042', 'position': 'RB', 'grid': (9, 2)},
    {'name': 'Cristian Romero', 'player_id': '355915', 'position': 'CB', 'grid': (7, 2)},
    {'name': 'Nicolás Otamendi', 'player_id': '54781', 'position': 'CB', 'grid': (3, 2)},
    {'name': 'Nicolás Tagliafico', 'player_id': '131225', 'position': 'LB', 'grid': (1, 2)},
    {'name': 'Rodrigo De Paul', 'player_id': '255901', 'position': 'CM', 'grid': (7, 3)},
    {'name': 'Enzo Fernández', 'player_id': '648195', 'position': 'CDM', 'grid': (5, 3)},
    {'name': 'Alexis Mac Allister', 'player_id': '534033', 'position': 'CM', 'grid': (3, 3)},
    {'name': 'Lionel Messi', 'player_id': '28003', 'position': 'RW', 'grid': (8, 4)},
    {'name': 'Julián Álvarez', 'player_id': '576024', 'position': 'ST', 'grid': (5, 5)},
    {'name': 'Ángel Di María', 'player_id': '45320', 'position': 'LW', 'grid': (2, 4)},
]

print(f"Defined {len(ARGENTINA_2022_FINAL)} players")
for p in ARGENTINA_2022_FINAL:
    print(f"  {p['position']:3} - {p['name']}")

Defined 11 players
  GK  - Emiliano Martínez
  RB  - Nahuel Molina
  CB  - Cristian Romero
  CB  - Nicolás Otamendi
  LB  - Nicolás Tagliafico
  CM  - Rodrigo De Paul
  CDM - Enzo Fernández
  CM  - Alexis Mac Allister
  RW  - Lionel Messi
  ST  - Julián Álvarez
  LW  - Ángel Di María


In [ ]:
def scrape_player_current_club(player_id, player_name):
    """
    Scrape player's profile to get their current club.
    For WC 2022 players, this should be close enough to their Dec 2022 club.
    """
    profile_url = f"{BASE_URL}/a/profil/spieler/{player_id}"
    
    soup = fetch_page(profile_url)
    if not soup:
        return {'club': 'Unknown', 'club_id': None}
    
    # Look for the player's current club in the header section
    # Usually shown prominently with club logo
    
    # Try finding club from the data-hauptpunkt section
    club_span = soup.find('span', class_='hauptpunkt')
    if club_span:
        club_link = club_span.find('a')
        if club_link:
            return {
                'club': club_link.text.strip(),
                'club_id': None
            }
    
    # Alternative: look for any club link in the info section
    info_table = soup.find('div', class_='info-table')
    if info_table:
        club_link = info_table.find('a', href=re.compile(r'/verein/\d+'))
        if club_link:
            return {
                'club': club_link.text.strip(),
                'club_id': None
            }
    
    # Fallback: find any club link
    all_club_links = soup.find_all('a', href=re.compile(r'/startseite/verein/\d+'))
    for link in all_club_links:
        text = link.text.strip()
        if text and len(text) > 2:
            return {'club': text, 'club_id': None}
    
    return {'club': 'Unknown', 'club_id': None}

# Test with one player
# test = scrape_player_current_club('28003', 'Lionel Messi')
# print(f"Messi's club: {test}")

In [9]:
# Since scraping can be slow and rate-limited, let's use known data for now
# These are the clubs each Argentina player was at during WC 2022

ARGENTINA_2022_CLUBS = {
    '111873': 'Aston Villa',        # Emiliano Martínez
    '424042': 'Atlético Madrid',    # Nahuel Molina
    '355915': 'Tottenham',          # Cristian Romero
    '54781': 'Benfica',             # Nicolás Otamendi
    '131225': 'Lyon',               # Nicolás Tagliafico
    '255901': 'Atlético Madrid',    # Rodrigo De Paul
    '648195': 'Benfica',            # Enzo Fernández
    '534033': 'Brighton',           # Alexis Mac Allister
    '28003': 'Paris Saint-Germain', # Lionel Messi
    '576024': 'Manchester City',    # Julián Álvarez
    '45320': 'Juventus',            # Ángel Di María
}

# Add club info to our player data
for player in ARGENTINA_2022_FINAL:
    player['club'] = ARGENTINA_2022_CLUBS.get(player['player_id'], 'Unknown')

print("Argentina 2022 WC Final Starting XI with clubs:")
print("="*60)
for p in ARGENTINA_2022_FINAL:
    print(f"{p['position']:3} | {p['name']:<22} | {p['club']}")

Argentina 2022 WC Final Starting XI with clubs:
GK  | Emiliano Martínez      | Aston Villa
RB  | Nahuel Molina          | Atlético Madrid
CB  | Cristian Romero        | Tottenham
CB  | Nicolás Otamendi       | Benfica
LB  | Nicolás Tagliafico     | Lyon
CM  | Rodrigo De Paul        | Atlético Madrid
CDM | Enzo Fernández         | Benfica
CM  | Alexis Mac Allister    | Brighton
RW  | Lionel Messi           | Paris Saint-Germain
ST  | Julián Álvarez         | Manchester City
LW  | Ángel Di María         | Juventus


## Step 5: Generate Quiz JSON

In [10]:
def generate_quiz_json(players, tournament, match_info, national_team, opponent, formation, match_date):
    """
    Generate a quiz JSON file from player data.
    """
    quiz = {
        'tournament': tournament,
        'match': match_info,
        'match_date': match_date,
        'national_team': national_team,
        'opponent': opponent,
        'formation': formation,
        'players': []
    }
    
    for p in players:
        quiz['players'].append({
            'position': p['position'],
            'name': p['name'],
            'club': p['club'],
            'grid_position': list(p['grid'])
        })
    
    return quiz

# Generate quiz for Argentina 2022 Final
quiz_json = generate_quiz_json(
    players=ARGENTINA_2022_FINAL,
    tournament='World Cup 2022',
    match_info='Final',
    national_team='Argentina',
    opponent='France',
    formation='4-3-3',
    match_date='2022-12-18'
)

print(json.dumps(quiz_json, indent=2))

{
  "tournament": "World Cup 2022",
  "match": "Final",
  "match_date": "2022-12-18",
  "national_team": "Argentina",
  "opponent": "France",
  "formation": "4-3-3",
  "players": [
    {
      "position": "GK",
      "name": "Emiliano Mart\u00ednez",
      "club": "Aston Villa",
      "grid_position": [
        5,
        1
      ]
    },
    {
      "position": "RB",
      "name": "Nahuel Molina",
      "club": "Atl\u00e9tico Madrid",
      "grid_position": [
        9,
        2
      ]
    },
    {
      "position": "CB",
      "name": "Cristian Romero",
      "club": "Tottenham",
      "grid_position": [
        7,
        2
      ]
    },
    {
      "position": "CB",
      "name": "Nicol\u00e1s Otamendi",
      "club": "Benfica",
      "grid_position": [
        3,
        2
      ]
    },
    {
      "position": "LB",
      "name": "Nicol\u00e1s Tagliafico",
      "club": "Lyon",
      "grid_position": [
        1,
        2
      ]
    },
    {
      "position": "CM",
      "na

In [11]:
# Save the quiz JSON
output_path = '../quizzes/starting11/preloaded/wc2022_final_argentina.json'

import os
os.makedirs(os.path.dirname(output_path), exist_ok=True)

with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(quiz_json, f, indent=2, ensure_ascii=False)

print(f"Quiz saved to: {output_path}")

Quiz saved to: ../quizzes/starting11/preloaded/wc2022_final_argentina.json


## Step 6: Mock Visualization

Create an HTML visualization to see if our data works for the game UI.

In [12]:
def generate_pitch_html(quiz_data):
    """
    Generate an HTML visualization of the formation with club badges.
    """
    # Map grid positions to CSS positions
    # Grid: x=1-10 (left to right), y=1-5 (GK to forwards)
    
    players_html = ""
    for p in quiz_data['players']:
        x, y = p['grid_position']
        # Convert grid to percentage positions
        left = (x / 10) * 100
        # Y is inverted: 1=bottom (GK), 5=top (forwards)
        top = 100 - (y / 5) * 80 - 10  # Leave margins
        
        players_html += f'''
        <div class="player" style="left: {left}%; top: {top}%;">
            <div class="badge">{p['club'][:3].upper()}</div>
            <div class="club-name">{p['club']}</div>
        </div>
        '''
    
    html = f'''
    <!DOCTYPE html>
    <html>
    <head>
        <style>
            body {{
                font-family: 'Press Start 2P', monospace, sans-serif;
                background: #1a1a2e;
                color: #facc15;
                display: flex;
                flex-direction: column;
                align-items: center;
                padding: 20px;
            }}
            h1 {{
                font-size: 14px;
                margin-bottom: 10px;
            }}
            .pitch {{
                width: 500px;
                height: 700px;
                background: linear-gradient(to bottom, 
                    #2d5a27 0%, #2d5a27 49%, 
                    #ffffff 49%, #ffffff 51%,
                    #2d5a27 51%, #2d5a27 100%);
                border: 4px solid #fff;
                border-radius: 8px;
                position: relative;
                box-shadow: 0 0 30px rgba(0,0,0,0.5);
            }}
            .pitch::before {{
                content: '';
                position: absolute;
                top: 50%;
                left: 50%;
                transform: translate(-50%, -50%);
                width: 120px;
                height: 120px;
                border: 3px solid rgba(255,255,255,0.5);
                border-radius: 50%;
            }}
            .player {{
                position: absolute;
                transform: translate(-50%, -50%);
                text-align: center;
            }}
            .badge {{
                width: 50px;
                height: 50px;
                background: #fff;
                border-radius: 50%;
                display: flex;
                align-items: center;
                justify-content: center;
                font-size: 10px;
                font-weight: bold;
                color: #1a1a2e;
                border: 3px solid #facc15;
                box-shadow: 0 2px 10px rgba(0,0,0,0.3);
                margin: 0 auto;
            }}
            .club-name {{
                font-size: 8px;
                margin-top: 4px;
                color: #fff;
                text-shadow: 1px 1px 2px #000;
                max-width: 80px;
                white-space: nowrap;
                overflow: hidden;
                text-overflow: ellipsis;
            }}
            .info {{
                margin-top: 20px;
                text-align: center;
            }}
            .answer {{
                font-size: 20px;
                margin-top: 20px;
                padding: 15px 30px;
                background: #facc15;
                color: #1a1a2e;
                border-radius: 8px;
            }}
            .meta {{
                font-size: 10px;
                color: #888;
                margin-top: 10px;
            }}
        </style>
    </head>
    <body>
        <h1>STARTING11 - GUESS THE NATIONAL TEAM</h1>
        <div class="pitch">
            {players_html}
        </div>
        <div class="info">
            <div class="meta">{quiz_data['tournament']} | {quiz_data['match']} | {quiz_data['formation']}</div>
            <div class="answer">🇦🇷 {quiz_data['national_team']}</div>
        </div>
    </body>
    </html>
    '''
    
    return html

# Generate the visualization
viz_html = generate_pitch_html(quiz_json)

# Save to file
with open('starting11_mock.html', 'w') as f:
    f.write(viz_html)

print("Mock visualization saved to: starting11_mock.html")
print("Open this file in a browser to see the result!")

Mock visualization saved to: starting11_mock.html
Open this file in a browser to see the result!


In [13]:
# Display inline in notebook (simplified version)
from IPython.display import HTML

# Create a simplified inline version
inline_html = '''
<div style="font-family: monospace; background: #1a1a2e; padding: 20px; border-radius: 10px;">
    <div style="color: #facc15; text-align: center; margin-bottom: 15px; font-weight: bold;">
        🏆 STARTING11 - World Cup 2022 Final
    </div>
    <div style="background: #2d5a27; padding: 20px; border-radius: 8px; border: 2px solid #fff;">
        <pre style="color: #fff; text-align: center; font-size: 11px; line-height: 2.5;">
                    [JUV]                      
           Di María                           
                                              
                        [MCI]        [PSG]    
                       Álvarez       Messi    
                                              
        [BRI]         [BEN]         [ATM]     
     Mac Allister    Fernández     De Paul    
                                              
   [LYO]    [BEN]           [TOT]    [ATM]    
 Tagliafico Otamendi       Romero   Molina    
                                              
                      [AVL]                   
                    Martínez                  
        </pre>
    </div>
    <div style="text-align: center; margin-top: 15px;">
        <span style="background: #facc15; color: #1a1a2e; padding: 10px 20px; border-radius: 5px; font-weight: bold;">
            🇦🇷 ARGENTINA
        </span>
    </div>
</div>
'''

display(HTML(inline_html))

## Summary

We've created:
1. **Scraper functions** - For extracting lineup data from Transfermarkt
2. **Quiz JSON generator** - Outputs the data in the format needed for Starting11
3. **Mock visualization** - HTML preview of how the game will look

**Next steps:**
- Test the actual scraper on more matches
- Add proper club badge images
- Build the Flask blueprint with the game logic